## Implemented with BiLSTM
1. Dropout -> Regularization
2. Class Weights -> Imbalance handling
3. Early Stopping -> Generalization
4. Hyperparameter Tuning -> Dimensions

In [15]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report

torch.manual_seed(0)

In [16]:
# Read the files

def read_file(path):

     data = []
     current_words = []
     current_tags = []

     for line in open(path, encoding='utf-8'):
          line = line.strip()

          if line:
               if line[0] == '#':
                    continue
               tok = line.split('\t')

               current_words.append(tok[1])
               current_tags.append(tok[2])

          else:
               if current_words:
                    data.append((current_words, current_tags))

               current_words = []
               current_tags = []

     if current_tags != []:
          data.append((current_words, current_tags))

     return data

In [ ]:
train_data = read_file('en_ewt-ud-train.iob2')
dev_data = read_file('en_ewt-ud-dev.iob2')

In [18]:
print(len(train_data))
print(len(dev_data))

12543
2001


In [19]:
# Prepare data for use in pytorch

class Vocab():
     def __init__(self):
          self.word2idx = {'<PAD>': 0, '<UNK>': 1} #dict to look up a number usng a word
          self.idx2word = ['<PAD>', '<UNK>'] #list to look up word using a number (index)

     def getIdx(self, word, add=False): #when training -> add=True
          if word not in self.word2idx:
               if add: #during training add the word
                    self.word2idx[word] = len(self.idx2word)
                    self.idx2word.append(word)
               else: #during dev/test return unk/pad id
                    return self.word2idx.get('<UNK>', 0) #if unknown
          #return the word id if already in dict
          return self.word2idx[word]

     def getWord(self, idx):
          return self.idx2word[idx]

word_vocab = Vocab() #turn words into numbers
label_vocab = Vocab() #turn IOB2 tags into numbers

max_len = max([len(x[0]) for x in train_data])

In [20]:
print(max_len)

159


In [21]:
def prepare_data(data, word_vocab, label_vocab, max_len, is_train = False):
     all_inputs_ids = []
     all_label_ids = []

     for words, labels in data:
          #numericalization, ie building the dictionary. during training
          word_ids = [word_vocab.getIdx(w, add=is_train) for w in words]
          label_ids = [label_vocab.getIdx(l, add=is_train) for l in labels]

          #every sentence must be the same length, add padding if sentence shorter than max len
          pad_len = max_len - len(word_ids)

          word_ids += [word_vocab.word2idx['<PAD>']] * pad_len
          label_ids += [label_vocab.word2idx['<PAD>']] * pad_len

          #truncate if by some chance any sentence exceeds max len
          all_inputs_ids.append(word_ids[:max_len])
          all_label_ids.append(label_ids[:max_len])

     #convert and return lists of lists into pytorch tensors
     return torch.tensor(all_inputs_ids), torch.tensor(all_label_ids)

train_inputs, train_labels = prepare_data(train_data, word_vocab, label_vocab, max_len, is_train=True)
dev_inputs, dev_labels = prepare_data(dev_data, word_vocab, label_vocab, max_len, is_train=False)

print(f"Train shape: {train_inputs.shape}") #should be [12543, 159]
print(f"Label shape: {train_labels.shape}") #should be [12543, 159]

Train shape: torch.Size([12543, 159])
Label shape: torch.Size([12543, 159])


In [22]:
# Convert data into batches
batch_size = 32

#wrap the tensors from prepare_data() into dataset object
train_dataset = TensorDataset(train_inputs, train_labels)
dev_dataset = TensorDataset(dev_inputs, dev_labels)

#create the loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size , shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size , shuffle=False)

In [ ]:
# Build the model

class LSTM(nn.Module):

     def __init__(self, vocab_size, embed_dim, hidden_dim, num_tags):
          super(LSTM, self).__init__()
          self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
          self.lstm = nn.LSTM(embed_dim, hidden_dim//2, batch_first=True, bidirectional=True)
          self.linear = nn.Linear(hidden_dim, num_tags)
          self.dropout = nn.Dropout(0.2)

     def forward(self, inputData):
          word_vectors = self.embedding(inputData)
          word_vectors = self.dropout(word_vectors)
          lstm_out, _ = self.lstm(word_vectors)
          dropped_out = self.dropout(lstm_out)
          tag_scores = self.linear(dropped_out)

          return tag_scores


EMBED_DIM = 100 #size of each word vector
HIDDEN_DIM = 64 #size of the hidden state
LEARNING_RATE = 0.001
EPOCHS = 10

model = LSTM(vocab_size=len(word_vocab.idx2word),
             embed_dim=EMBED_DIM,
             hidden_dim = HIDDEN_DIM,
             num_tags=len(label_vocab.idx2word))

optimizer = torch.optim.Adam(model.parameters(), lr = LEARNING_RATE, weight_decay=1e-5)

#handle class imbalance caused by 'O'. Give 'O' small(0.1) weight and the other labels a bigger(1.0) weight
weights = torch.ones(len(label_vocab.idx2word))
o_idx = label_vocab.word2idx['O']
weights[o_idx] = 0.1

loss_function = torch.nn.CrossEntropyLoss(ignore_index=0, weight=weights, reduction='mean')

In [ ]:
# Train the model

def train_epoch(model, train_loader, optimizer, loss_function):
     model.train()
     total_loss = 0

     #loop over batches
     for batch_x, batch_y in train_loader:
          optimizer.zero_grad()

          #calculate loss
          pred_values = model(batch_x)
          logits = pred_values.view(-1, len(label_vocab.idx2word))
          targets = batch_y.view(-1)
          loss = loss_function(logits, targets)

          #update
          loss.backward()
          optimizer.step()

          total_loss += loss.item()

     return total_loss / len(train_loader)

In [ ]:
# Validate model

def validate_epoch(model, dev_loader):
     model.eval()
     all_preds = []
     all_labels = []
     total_loss = 0

     with torch.no_grad():
          for batch_x, batch_y in dev_loader:
               output = model(batch_x)

               #track loss
               logits = output.view(-1, len(label_vocab.idx2word))
               targets = batch_y.view(-1)
               loss = loss_function(logits, targets)
               total_loss += loss.item()

               #get the highest score index for each word
               preds = torch.argmax(output, dim=-1)

               mask = (batch_y != 0)

               all_preds.extend(preds[mask].tolist())
               all_labels.extend(batch_y[mask].tolist())

     #map the numbers back to their tag names
     #skip first tag if PAD
     target_names = label_vocab.idx2word[1:]
     presented_labels = sorted(list(set(all_labels)))
     target_names = [label_vocab.idx2word[i] for i in presented_labels]

     #detailed classification report
     class_report = classification_report(all_labels, all_preds, target_names=target_names)
     return class_report, total_loss / len(dev_loader)

In [ ]:
#implement early stop
best_val_loss = float('inf')
patience = 3 #how many epochs before giving up
patience_counter = 0 #track how many epochs since last improvement

for epoch in range(EPOCHS):
     #one full training pass
     avg_train_loss = train_epoch(model, train_loader, optimizer, loss_function,)
     class_rep, avg_dev_loss = validate_epoch(model, dev_loader)
     print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train Loss: {float(avg_train_loss):.4f} | Validation Loss: {float(avg_dev_loss):.4f}")

     #check improvement
     if avg_dev_loss < best_val_loss:
          best_val_loss = avg_dev_loss
          patience_counter = 0

     else:
          patience_counter += 1

     #stop if ran out of patience
     if patience_counter >= patience:
          print('Early Stop')
          break

#model.load_state_dict(torch.load(PATH))

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch   1/10 | Train Loss: 1.1156 | Validation Loss: 0.9110
Epoch   2/10 | Train Loss: 0.7486 | Validation Loss: 0.7734
Epoch   3/10 | Train Loss: 0.5658 | Validation Loss: 0.6687
Epoch   4/10 | Train Loss: 0.4251 | Validation Loss: 0.6539
Epoch   5/10 | Train Loss: 0.3117 | Validation Loss: 0.6921
Epoch   6/10 | Train Loss: 0.2314 | Validation Loss: 0.6890
Epoch   7/10 | Train Loss: 0.1724 | Validation Loss: 0.7054
Early Stop


In [28]:
print(class_rep)

              precision    recall  f1-score   support

           O       0.97      0.99      0.98     23653
       B-LOC       0.73      0.60      0.66       399
       I-LOC       0.47      0.43      0.45       148
       B-PER       0.71      0.50      0.59       343
       B-ORG       0.47      0.34      0.39       224
       I-ORG       0.45      0.38      0.41       186
       I-PER       0.65      0.51      0.57       196

    accuracy                           0.96     25149
   macro avg       0.64      0.54      0.58     25149
weighted avg       0.95      0.96      0.96     25149



In [ ]:
# Test data
test_data = read_file('en_ewt-ud-test-masked.iob2')

def predict(model, test_data):
     model.eval()
     with open('en-ewt-predictions.iob2', 'w') as file:
        for words, masked_tags in test_data:
            #convert words to indices
            word_indices = [word_vocab.getIdx(w, add=False) for w in words]
            words_tensor = torch.tensor(word_indices)
            #run model
            with torch.no_grad():
                output = model(words_tensor)

                #get the prediction = highest score for each word
                preds = torch.argmax(output, dim=-1)

                #convert indices to tag/label names
                pred_tags = [label_vocab.idx2word[p.item()] for p in preds]

                #write word + preditced tag to file
                for i, (word, pred_tag) in enumerate(zip(words, pred_tags)):
                    file.write(f"{i}\t{word}\t{pred_tag}\n")

                #blank line between sentences
                file.write('\n')

predict(model, test_data)